# PRUEBAS DE CLUSTERING APLICADO AL DATASET ELECTORAL LAPOP 2023

### INICIALIZACIÓN

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.clustering.traditional import KMedoids, HierarchicalClustering, DBSCAN
from src.evaluation import calculate_metrics
from src.utils.constants import PROCESSED_DATA_PATH

print("="*70)
print("TEST SIMPLE DE ALGORITMOS DE CLUSTERING")
print("="*70)

# Cargar dataset preprocesado
dataset_path = f"{PROCESSED_DATA_PATH}"
df = pd.read_csv(dataset_path)

print(f"\n✓ Dataset cargado:")
print(f"  Dimensiones: {df.shape[0]} registros × {df.shape[1]} variables")
print(f"  Missings: {df.isnull().sum().sum()}")
print(f"\nPrimeras 3 filas:")
print(df.head(3))
print("="*70)

### APLICACIÓN DE LOS ALGORITMOS

In [ ]:
print("\n--- TEST K-MEDOIDS ---\n")

# Buscar k óptimo con método del codo + silhouette
print("Buscando k óptimo...")
k_range = range(2, 12)
resultados_k = []

for k in k_range:
    km_temp = KMedoids(n_clusters=k, random_state=42)
    km_temp.fit(df)
    metrics_temp = calculate_metrics(df, km_temp.labels_)
    resultados_k.append({
        'k': k,
        'inercia': km_temp.inertia_,
        'silhouette': metrics_temp['silhouette'],
        'davies_bouldin': metrics_temp['davies_bouldin']
    })
    print(f"  k={k}: Inercia={km_temp.inertia_:.2f} | "
          f"Sil={metrics_temp['silhouette']:.4f} | "
          f"DB={metrics_temp['davies_bouldin']:.4f}")

# Buscar k óptimo: mayor silhouette
df_k = pd.DataFrame(resultados_k)
k_optimo_sil = df_k.loc[df_k['silhouette'].idxmax(), 'k']
k_optimo_db  = df_k.loc[df_k['davies_bouldin'].idxmin(), 'k']

print(f"\n  k óptimo por Silhouette:      k={k_optimo_sil}")
print(f"  k óptimo por Davies-Bouldin:  k={k_optimo_db}")

# Entrenar con k óptimo
k_final = int(k_optimo_sil)
print(f"\nEntrenando con k={k_final}...")
kmedoids = KMedoids(n_clusters=k_final, random_state=42)
kmedoids.fit(df)

# Distribución y métricas
print("\nDistribución:")
print(kmedoids.get_cluster_distribution())
print("\nMétricas:")
metrics_km = calculate_metrics(df, kmedoids.labels_)

print("="*70)

In [ ]:
print("\n--- TEST CLUSTERING JERÁRQUICO ---\n")

# PASO 1: Busqueda del mejor linkage (usando n_clusters provisional para comparar para linkages)
print("Paso 1: Comparando métodos de linkage...")
linkage_methods = ['ward', 'complete', 'average', 'single']
resultados_linkage = []

for linkage in linkage_methods:
    hc_temp = HierarchicalClustering(n_clusters=4, linkage=linkage)
    hc_temp.fit(df)
    metrics_temp = calculate_metrics(df, hc_temp.labels_)
    resultados_linkage.append({
        'linkage': linkage,
        'silhouette': metrics_temp['silhouette']
    })
    print(f"  {linkage:10s}: Sil={metrics_temp['silhouette']:.4f}")

mejor_linkage = max(resultados_linkage, key=lambda x: x['silhouette'])['linkage']
print(f"\n  Mejor linkage: '{mejor_linkage}'")

# PASO 2: Con el mejor linkage, buscar n_clusters óptimo
print(f"\nPaso 2: Buscando n_clusters óptimo con linkage='{mejor_linkage}'...")
resultados_n = []

for n in range(2, 12):
    hc_temp = HierarchicalClustering(n_clusters=n, linkage=mejor_linkage)
    hc_temp.fit(df)
    metrics_temp = calculate_metrics(df, hc_temp.labels_)
    resultados_n.append({
        'n_clusters': n,
        'silhouette': metrics_temp['silhouette'],
        'davies_bouldin': metrics_temp['davies_bouldin']
    })
    print(f"  n={n}: Sil={metrics_temp['silhouette']:.4f} | "
          f"DB={metrics_temp['davies_bouldin']:.4f}")

df_n = pd.DataFrame(resultados_n)
n_optimo = int(df_n.loc[df_n['silhouette'].idxmax(), 'n_clusters'])
print(f"\n  n_clusters óptimo: {n_optimo}")

# PASO 3: Entrenar con los parámetros encontrados
print(f"\nPaso 3: Entrenando con n_clusters={n_optimo}, linkage='{mejor_linkage}'...")
hierarchical = HierarchicalClustering(n_clusters=n_optimo, linkage=mejor_linkage)
hierarchical.fit(df)

print("\nDistribución:")
print(hierarchical.get_cluster_distribution())
print("\nMétricas finales:")
metrics_hc = calculate_metrics(df, hierarchical.labels_)

print("="*70)

In [ ]:
print("\n--- DENDROGRAMA - CLUSTERING JERÁRQUICO ---\n")

fig1 = hierarchical.plot_dendrogram(
    figsize=(16, 7),
    truncate_mode=None,
    save_path='../../figures/dendrograma_completo.png'
)
plt.show()

fig2 = hierarchical.plot_dendrogram(
    figsize=(14, 6),
    truncate_mode='lastp',
    p=30,
    save_path='../../figures/dendrograma_truncado.png'
)

ax = plt.gca()
colores = ['red', 'orange', 'green']
alturas = [0.65, 0.50, 0.35] 

for altura, color, n in zip(alturas, colores, [2, 3, 4]):
    ax.axhline(
        y=altura * ax.get_ylim()[1],
        color=color,
        linestyle='--',
        linewidth=1.5,
        label=f'Corte → {n} clusters'
    )

ax.legend(loc='upper right')
plt.show()

print("\nInterpretación:")
print(f"  Linkage usado: {hierarchical.linkage}")
print(f"  n_clusters actual: {hierarchical.n_clusters_}")
print(f"  Silhouette: {metrics_hc['silhouette']:.4f}")
print("\n  Las líneas de corte muestran dónde cortar el dendrograma")
print("  para obtener 2, 3 o 4 clusters.")
print("  Los saltos grandes en altura = clusters naturales en los datos.")

In [ ]:
print("\n--- TEST DBSCAN ---\n")

# DBSCAN con parámetros ajustados para datos de alta dimensionalidad
print("Probando diferentes configuraciones de DBSCAN...\n")

configs = [
    {'eps': eps, 'min_samples': ms}
    for eps in [1.0, 2.0, 3.0, 4.0, 5.0]
    for ms in [3, 5, 10]
]

mejor_config = None
mejor_sil = -1

for cfg in configs:
    db_temp = DBSCAN(**cfg)
    db_temp.fit(df)
    if db_temp.n_clusters_ >= 2:
        mask = db_temp.labels_ != -1
        sil = calculate_metrics(df[mask], db_temp.labels_[mask])['silhouette']
        print(f"  eps={cfg['eps']}, min_samples={cfg['min_samples']}: "
              f"{db_temp.n_clusters_} clusters, Sil={sil:.4f}")
        if sil > mejor_sil:
            mejor_sil = sil
            mejor_config = cfg
            dbscan = db_temp
    else:
        print(f"  eps={cfg['eps']}, min_samples={cfg['min_samples']}: sin clusters")

if mejor_config:
    print(f"\n✓ Mejor configuración: {mejor_config} → Sil={mejor_sil:.4f}")
else:
    print("⚠ Ninguna configuración generó clusters válidos")

print(f"\nConfiguración final: eps={dbscan.eps}, min_samples={dbscan.min_samples}")
print(f"Clusters identificados: {dbscan.n_clusters_}")
print(f"Puntos de ruido: {dbscan.n_noise_points_}")
print(f"\nTamaños:")
print(dbscan.get_cluster_sizes())

# Métricas (solo si hay suficientes clusters)
if dbscan.n_clusters_ >= 2:
    # Filtrar noise
    mask = dbscan.labels_ != -1
    df_clean = df[mask]
    labels_clean = dbscan.labels_[mask]
    
    print("\nMétricas (sin ruido):")
    metrics_db = calculate_metrics(df_clean, labels_clean)
else:
    print("\n⚠ DBSCAN no generó clusters válidos con ninguna configuración")
    print("  Esto puede ocurrir en datos de alta dimensionalidad")
    print("  DBSCAN requiere ajuste manual de parámetros según el dataset")
    metrics_db = None

print("="*70)

### METRICAS Y RESUMEN

In [ ]:
print("\n--- COMPARACIÓN DE ALGORITMOS ---\n")

# Tabla comparativa
comparison_data = {
    'Algoritmo': ['K-Medoids', 'Hierarchical', 'DBSCAN'],
    'Clusters': [kmedoids.n_clusters_, hierarchical.n_clusters_, dbscan.n_clusters_],
    'Silhouette': [
        metrics_km['silhouette'], 
        metrics_hc['silhouette'],
        metrics_db['silhouette'] if metrics_db else np.nan
    ],
    'Davies-Bouldin': [
        metrics_km['davies_bouldin'],
        metrics_hc['davies_bouldin'],
        metrics_db['davies_bouldin'] if metrics_db else np.nan
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n" + "="*70)
print("✓ TEST COMPLETADO")
print("="*70)

In [ ]:
print("\n--- VISUALIZACIÓN DE CLUSTERS ---\n")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 5)

# Reducir dimensionalidad a 2D con PCA
print("Aplicando PCA para reducir a 2 dimensiones...")
pca = PCA(n_components=2, random_state=42)
df_2d = pca.fit_transform(df)

print(f"✓ Varianza explicada: {pca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")

# Crear figura con 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ====== K-Medoids ======
scatter1 = axes[0].scatter(
    df_2d[:, 0], 
    df_2d[:, 1], 
    c=kmedoids.labels_, 
    cmap='viridis', 
    alpha=0.6,
    s=30,
    edgecolors='black',
    linewidth=0.5
)
axes[0].set_title(f'K-Medoids (k={kmedoids.n_clusters_})\nSilhouette: {metrics_km["silhouette"]:.3f}', 
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# ====== Hierarchical ======
scatter2 = axes[1].scatter(
    df_2d[:, 0], 
    df_2d[:, 1], 
    c=hierarchical.labels_, 
    cmap='plasma', 
    alpha=0.6,
    s=30,
    edgecolors='black',
    linewidth=0.5
)
axes[1].set_title(f'Hierarchical ({hierarchical.n_clusters_} clusters)\nSilhouette: {metrics_hc["silhouette"]:.3f}', 
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

# ====== DBSCAN ======
if dbscan.n_clusters_ >= 2:
    # DBSCAN con clusters válidos
    scatter3 = axes[2].scatter(
        df_2d[:, 0], 
        df_2d[:, 1], 
        c=dbscan.labels_, 
        cmap='coolwarm', 
        alpha=0.6,
        s=30,
        edgecolors='black',
        linewidth=0.5
    )
    title = f'DBSCAN ({dbscan.n_clusters_} clusters, {dbscan.n_noise_points_} noise)\nSilhouette: {metrics_db["silhouette"]:.3f}'
else:
    # DBSCAN sin clusters (todo ruido)
    scatter3 = axes[2].scatter(
        df_2d[:, 0], 
        df_2d[:, 1], 
        c='gray', 
        alpha=0.3,
        s=30,
        edgecolors='black',
        linewidth=0.5
    )
    title = f'DBSCAN (Sin clusters)\nTodos los puntos = ruido'

axes[2].set_title(title, fontsize=12, fontweight='bold')
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
if dbscan.n_clusters_ >= 2:
    plt.colorbar(scatter3, ax=axes[2], label='Cluster')

plt.tight_layout()
plt.savefig('../../figures/comparacion_clusters_2d.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualización guardada en: figures/comparacion_clusters_2d.png")
print("="*70)